# NLP Real-Time Text Classification

This notebook loads the saved best NLP model and runs inference on user-entered text. It does not train anything. The input text is cleaned, tokenized, padded, and passed through the trained model for forward prediction.


## 1. Load model, tokenizer, and preprocessing config


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json

from src.paths import NLP
from src.nlp.data import TARGET_NAMES
from src.nlp.text import LemmaNormalizer, clean_text

registry_path = NLP.tables / "nlp_experiment_registry.csv"
if not registry_path.exists():
    raise FileNotFoundError(f"Missing registry: {registry_path}. Run the NLP training notebook first.")

registry = pd.read_csv(registry_path)
model_name = registry.iloc[0]["experiment_name"]
model_path = NLP.models / f"{model_name}.keras"
if not model_path.exists():
    raise FileNotFoundError(f"Missing model: {model_path}")

tokenizer_path = NLP.processed / "tokenizer_lemmatized.json"
config_path = NLP.tables / "preprocessing_config.csv"
if not tokenizer_path.exists():
    raise FileNotFoundError(f"Missing tokenizer: {tokenizer_path}")
if not config_path.exists():
    raise FileNotFoundError(f"Missing preprocessing config: {config_path}")

tokenizer = tokenizer_from_json(tokenizer_path.read_text(encoding="utf-8"))
config = pd.read_csv(config_path)
max_len = int(config.loc[config["variant"] == "lemmatized", "max_len"].iloc[0])
normalizer = LemmaNormalizer()
model = tf.keras.models.load_model(model_path, compile=False)

print(f"Loaded model: {model_name}")
print(f"Max sequence length: {max_len}")


I0000 00:00:1778065727.497937   13730 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778065727.892973   13730 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778065729.926497   13730 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778065735.254225   13730 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device

Loaded model: lstm_glove_twitter_50_small_regularized_weighted_lemmatized
Max sequence length: 13


## 2. Prediction function


In [2]:
def classify_text(text: str) -> pd.DataFrame:
    cleaned = clean_text(text, normalizer=normalizer)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(sequence, maxlen=max_len, padding="post", truncating="post")
    probabilities = model.predict(padded, verbose=0)[0]

    result = pd.DataFrame({
        "label": TARGET_NAMES,
        "probability": probabilities,
    }).sort_values("probability", ascending=False).reset_index(drop=True)

    print("Raw text:", text)
    print("Cleaned text:", cleaned)
    print("Prediction:", result.loc[0, "label"])
    return result

classify_text("I cannot believe people are saying this online.")


Raw text: I cannot believe people are saying this online.
Cleaned text: believe people saying online
Prediction: neither


,label,probability
0,neither,0.962660
1,hate_speech,0.023323
2,offensive_language,0.014017


In [3]:
text = input("Enter a tweet: ")
classify_text(text)


Enter a tweet:  college is fun


Raw text: college is fun
Cleaned text: college fun
Prediction: neither


,label,probability
0,neither,0.975652
1,hate_speech,0.014326
2,offensive_language,0.010023
